In [0]:
%run "../00_config"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone

##Load Bronze products

In [0]:
df_bronze = spark.table(BRONZE_PRODUCTS)

##Standardize category names

In [0]:
# Build category mapping dynamically from Bronze data
distinct_categories = [
    row["category"]
    for row in df_bronze
        .select("category")
        .distinct()
        .filter(F.col("category").isNotNull())
        .filter(F.col("category") != "")
        .collect()
    if row["category"]
]

# Build Spark expression: "mobile phones" → "Mobile Phones" and handle _ between words
category_expr = F.col("category")
for cat in distinct_categories:
    category_expr = F.when(
        F.lower(F.col("category")) == cat.lower(),
        cat.replace("_", " ").title()
    ).otherwise(category_expr)

df_silver = df_bronze.withColumn("category", category_expr)

##Clean and transform

In [0]:
df_silver = (df_silver
    # Cast types
    .withColumn("current_price_sar",         F.col("current_price_sar").cast(DoubleType()))
    .withColumn("before_price_sar",           F.col("before_price_sar").cast(DoubleType()))
    .withColumn("rating",                     F.col("rating").cast(DoubleType()))
    .withColumn("position",                   F.col("position").cast(IntegerType()))
    .withColumn("total_reviews",
        F.when(F.col("total_reviews").cast(IntegerType()) == 0, F.lit(None))
        .otherwise(F.col("total_reviews").cast(IntegerType())
    )
)

    # Cast booleans
    .withColumn("is_amazon_choice",    F.col("is_amazon_choice").cast(BooleanType()))
    .withColumn("is_best_seller",      F.col("is_best_seller").cast(BooleanType()))
    .withColumn("is_prime",            F.col("is_prime").cast(BooleanType()))
    .withColumn("is_sponsored",        F.col("is_sponsored").cast(BooleanType()))

    # Replace empty strings with null
    .withColumn("bought_past_month",
        F.when(F.col("bought_past_month") == "", F.lit(None))
        .otherwise(F.col("bought_past_month"))
    )
    .withColumn("stock_warning",
        F.when(F.col("stock_warning") == "", F.lit(None))
        .otherwise(F.col("stock_warning"))
    )

    # Clean date
    .withColumn("_snapshot_date", (F.to_date(F.col("_snapshot_date"), "dd-MM-yyyy")))

    # Filter out products with no price or no ASIN
    .filter(F.col("asin") != "")
    .filter(F.col("asin").isNotNull())
    .filter(F.col("current_price_sar").isNotNull())
    .filter(F.col("current_price_sar") > 0)

    # Deduplicate — keep one row per ASIN per snapshot date
    .dropDuplicates(["asin", "_snapshot_date"])

    # Add ingestion metadata
    .withColumn("_silver_ingested_at", F.to_timestamp(F.lit(datetime.now(timezone.utc).isoformat())))

    # Final column selection
    .select(
        "asin",
        "product_title",
        "category",
        "current_price_sar",
        "before_price_sar",
        "rating",
        "total_reviews",
        "is_amazon_choice",
        "is_best_seller",
        "is_prime",
        "is_sponsored",
        "bought_past_month",
        "position",
        "stock_warning",
        "product_photo",
        "product_url",
        "_snapshot_date",
        "_silver_ingested_at"
    )
)

print(f"Silver rows after cleaning: {df_silver.count()}")
display(df_silver)

##Write to Silver

In [0]:
(df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_PRODUCT)
)

print(f"✅ {df_silver.count()} products written to {SILVER_PRODUCT}")

##Validate

In [0]:
df_check = spark.table(SILVER_PRODUCT)

print(f"Total rows        : {df_check.count()}")
print(f"Distinct ASINs    : {df_check.select('asin').distinct().count()}")
print(f"Categories        : {df_check.select('category').distinct().collect()}")
print(f"Null total_reviews: {df_check.filter(F.col('total_reviews').isNull()).count()}")
print(f"Price range       : {df_check.agg(F.min('current_price_sar'), F.max('current_price_sar')).collect()[0]}")
print(f"Nulls in price    : {df_check.filter(F.col('current_price_sar').isNull()).count()}")

df_check.select(
    "asin", "product_title", "category",
    "current_price_sar", "rating"
).show(5, truncate=40)

In [0]:
%sql
SELECT *
FROM saudi_retail_catalog.silver.silver_product;